# Chapter 4 exercises -- worked solutions

Full solutions to the four exercise stubs in `exercises/`. Each cell is
self-contained and runnable. Read the stub's docstring first; these are
the answers.

## Exercise 4.2 -- focal loss (gamma = 2)

Down-weight easy, frequent bins by `(1 - p_t) ** gamma` so the head
keeps learning rare bins. With `gamma = 0` it reduces to plain CE.

In [1]:
import torch
import torch.nn.functional as F


def focal_loss(logits, target_bins, gamma=2.0):
    n_bins = logits.shape[-1]
    ce = F.cross_entropy(
        logits.reshape(-1, n_bins), target_bins.reshape(-1),
        reduction="none",
    )
    p_t = torch.exp(-ce)
    return (((1 - p_t) ** gamma) * ce).mean()


torch.manual_seed(0)
logits = torch.randn(8, 16, 6, 256)
targets = torch.randint(0, 256, (8, 16, 6))
ce = F.cross_entropy(logits.reshape(-1, 256), targets.reshape(-1))
print("cross-entropy :", round(ce.item(), 4))
print("focal(gamma=0):", round(focal_loss(logits, targets, 0.0).item(), 4))
print("focal(gamma=2):", round(focal_loss(logits, targets).item(), 4))

cross-entropy : 6.0626
focal(gamma=0): 6.0626
focal(gamma=2): 6.0233


## Exercise 4.3 -- multimodal stress test

Fraction of (gripper, wrist) pairs in an incoherent quadrant: exactly
one joint in the top half of its range. A coherent (correlated) set
scores low; independent samples score ~0.5.

In [2]:
import numpy as np


def incoherent_fraction(pairs, n_bins=256):
    hi = np.asarray(pairs) >= (n_bins // 2)
    return float((hi[:, 0] ^ hi[:, 1]).mean())


rng = np.random.default_rng(0)
diag = rng.integers(0, 256, 300)
coherent = np.stack([diag, np.clip(diag + rng.integers(-4, 5, 300),
                     0, 255)], axis=1)
independent = np.stack([rng.integers(0, 256, 300),
                        rng.integers(0, 256, 300)], axis=1)
print("coherent    :", incoherent_fraction(coherent))
print("independent :", incoherent_fraction(independent))

coherent    : 0.016666666666666666
independent : 0.4866666666666667


## Exercise 4.4 -- per-dimension entropy diagnostic

Average the per-position softmax entropy over batch and horizon to get
one number per joint; a collapsed joint shows up as a short bar.

In [3]:
import numpy as np
from ch04.diagnostics import softmax_entropy


def per_dimension_entropy(logits):
    ent = softmax_entropy(logits)
    arr = ent.numpy() if hasattr(ent, "numpy") else np.asarray(ent)
    return arr.mean(axis=(0, 1))


rng = np.random.default_rng(0)
logits = rng.normal(0, 0.1, size=(8, 16, 6, 256))
logits[:, :, 5, 100] += 12.0  # collapse dim 5
print("per-dim entropy:", np.round(per_dimension_entropy(logits), 3))
print("uniform ref    :", round(float(np.log(256)), 3))

per-dim entropy: [5.54  5.54  5.54  5.54  5.54  0.021]
uniform ref    : 5.545


## Exercise 4.6 -- k-means binning (K = 256)

One 1-D k-means per dimension via a short Lloyd's loop, initialized on
quantiles. Encode is nearest-center assignment. Compare the mean
absolute round-trip error against the uniform tokenizer.

In [4]:
import numpy as np


def fit_kmeans_centers(actions, k=256, iters=25):
    n, d = actions.shape
    centers = np.zeros((d, k))
    for dim in range(d):
        col = actions[:, dim]
        cen = np.quantile(col, np.linspace(0, 1, k))
        for _ in range(iters):
            idx = np.argmin(np.abs(col[:, None] - cen[None, :]), axis=1)
            for j in range(k):
                m = col[idx == j]
                if m.size:
                    cen[j] = m.mean()
            cen = np.sort(cen)
        centers[dim] = cen
    return centers


def encode_kmeans(actions, centers):
    out = np.zeros(actions.shape, dtype=np.int64)
    for dim in range(actions.shape[-1]):
        out[..., dim] = np.argmin(
            np.abs(actions[..., dim, None] - centers[dim]), axis=-1)
    return out


# Small synthetic skewed column to keep the solution fast + offline.
rng = np.random.default_rng(0)
col = np.concatenate([
    rng.normal(-0.5, 0.05, (400, 6)),
    rng.normal(0.5, 0.05, (400, 6)),
], axis=0)
centers = fit_kmeans_centers(col, k=64, iters=10)
bins = encode_kmeans(col, centers)
recon = np.take_along_axis(centers.T, bins, axis=0)
print("k-means (K=64) mean abs error:", round(float(np.abs(recon - col).mean()), 5))

k-means (K=64) mean abs error: 0.00184
